# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all dataset elements by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s. This helps identify what data is accessible in the dataset using Croissant.

All Croissant entities must be referenced by their `@id`.

In [ ]:
# Examine all record sets in the dataset and their fields/columns

print("Available Record Sets (by @id):\n==============================")
record_set_ids = []
for record_set in getattr(metadata, 'record_set', []):
    rec_set_id = getattr(record_set, '@id', None)
    rec_set_name = getattr(record_set, 'name', '(No name)')
    print(f"- {rec_set_id} | name: {rec_set_name}")
    record_set_ids.append(rec_set_id)
    # List fields in the record set (by @id and name)
    fields = getattr(record_set, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"    - {field_id} | name: {field_name}")
        # List columns for the field (if any)
        columns = getattr(field, 'column', [])
        if not isinstance(columns, list):
            columns = [columns]
        if columns:
            print("      Columns:")
            for column in columns:
                col_id = getattr(column, '@id', None)
                col_name = getattr(column, 'name', None)
                print(f"        - {col_id} | name: {col_name}")
print("\nAll record_set @id values:", record_set_ids)

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

If there are multiple record sets, we extract each by their `@id` into a DataFrame.

In [ ]:
# Choose a record set to work with (update the @id as listed above)
if len(record_set_ids) == 0:
    raise Exception("No record sets found in this dataset.")

record_sets = record_set_ids
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from record_set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in record_set {record_set_id}: {df.columns.tolist()}")
    if not df.empty:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, outlier removal, and grouping, referencing all fields by their `@id`.

In this section, we select a numeric field (by `@id`) for demonstration. You should update field and group field `@id`s based on the dataset overview printed above.

In [ ]:
# --- Set these according to fields displayed in the overview (by their `@id`) ---
# Example ids (replace as needed):
example_record_set_id = record_set_ids[0]  # Use the first record set found by default
df = dataframes[example_record_set_id].copy()

# Try to auto-detect a numeric column by checking data types
numeric_candidate_ids = df.select_dtypes(include='number').columns.tolist()
if len(numeric_candidate_ids) == 0:
    raise Exception("No numeric fields found in the record set for EDA.")
numeric_field_id = numeric_candidate_ids[0]
print(f"Using numeric field (by @id): {numeric_field_id}")

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 0  # Example threshold

# Filter rows where numeric_field_id > threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (using @id field reference):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a candidate group field (prefer an object/categorical column not used above)
group_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Grouping by field (by @id): {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields, using their `@id` for axes. Replace field `@id`s with those relevant to your dataset as required.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

if group_candidates:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated step-by-step how to load, inspect, filter, and visualize a Croissant-structured dataset using the `mlcroissant` library. Data elements, including record sets and fields, were always referenced by their `@id` for consistency. Further investigations can be performed by adjusting field and record set `@id`s according to specific analysis needs.